# V40 DATA EXCHANGE — FULL GPU ARM (B1 crucible)
Bar: TS-val CE <= **2.2706** (v38e uniform-allocation control @ 13ep).

**Runtime must be T4 GPU** (preselected below — if it connects as CPU, Runtime > Change runtime type > T4).

Run Cell 1, then Cell 2, keep the tab open (~3h). Verdict prints at the end and lands in `/content/v40_report.json`.

In [ ]:
# ==== V40 COLAB CELL 1 — setup (paste into a T4 Colab notebook, run) ====
# 1) upload kaggle.json when prompted (kaggle.com -> Settings -> Create New Token)
import os, json, subprocess, sys
os.makedirs("/root/.kaggle", exist_ok=True)
from google.colab import files
up = files.upload()  # choose kaggle.json
with open("/root/.kaggle/kaggle.json","w") as f: f.write(up["kaggle.json"].decode())
os.chmod("/root/.kaggle/kaggle.json", 0o600)
subprocess.run([sys.executable,"-m","pip","install","-q","kaggle"],check=True)

# 2) fetch the four data datasets + the kernel SOURCE (all pre-pushed, no GPU quota involved)
for ds in ["albanchigozirim/tinystories-gate2-data","albanchigozirim/tinystories-gate3-data",
           "albanchigozirim/ag-news-v1","albanchigozirim/curriculum-v37-ckpt",
           "albanchigozirim/v40-exchange-src"]:
    subprocess.run(["kaggle","datasets","download","-d",ds,"--unzip","-p","/content/data"],check=True)
os.environ["V40_DATA"]="/content/data"
KERNEL="/content/data/kernel.py"
assert os.path.exists(KERNEL), "kernel.py missing from v40-exchange-src dataset"

# 3) ANCHOR + ASSERT protocol (paid in blood x5 — no silent misses):
src=open(KERNEL).read()
for tag in ['alloc_ledger.append','price_hist.append','FLOOR)','KAPPA_P',
            'CLEAR=10 if V40_SMOKE else 100','np.add.at(contrib','mkt_shares=ns']:
    assert tag in src, f"EXCHANGE PLUMBING MISSING: {tag}"
src=src.replace('os.environ.get("V40_SMOKE","1")=="1"','"0"=="1"')  # FULL ARM flip (env default is smoke)
assert '"0"=="1"' in src, "smoke->full switch failed (CLEAR=100 follows automatically)"
open(KERNEL,"w").write(src)
print("[assert] exchange plumbing verified; smoke->full switched; CLEAR=100")

# 4) wheel probe: Colab torch vs the cu12torch2.10 wheels (comparator fidelity:
#    v38e control 2.2706 ran on Kaggle's real-Mamba stack; we need REAL Mamba here)
import torch
print("colab torch:",torch.__version__,"| gpu:",torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
try:
    from mamba_ssm import Mamba; print("[probe] mamba-ssm ALREADY importable — skip install")
except Exception:
    r=subprocess.run([sys.executable,"-m","pip","install","--no-cache-dir","--no-deps",
      "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
      "https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"],
      capture_output=True,text=True)
    try:
        from mamba_ssm import Mamba; print("[probe] wheels OK on this torch")
    except Exception as e:
        print("[probe] WHEEL MISMATCH — run source build (~20min), then RE-RUN from cell 2:")
        print("   !pip install causal-conv1d==1.7.0 mamba-ssm==2.3.2.post1 --no-build-isolation")
        print("   detail:",str(e)[:200])
print("[setup] done — now run CELL 2 (the training run, ~3h on T4; per-epoch save is in the kernel)")


In [ ]:
!python /content/data/kernel.py 2>&1 | tee /content/v40_run.log

In [ ]:
import json, math
r = json.load(open('/content/v40_report.json'))
best = r.get('best_val_ce'); ctrl = 2.2706
led = r.get('alloc_ledger', []); ood = r.get('ood_probe', {}) or {}
print('B1 market<=control:', best, 'vs', ctrl, '->', 'PASS' if (best is not None and best<=ctrl) else 'FAIL')
print('B2 OOD:', ood.get('n_distinct'), 'distinct | ood_ok', ood.get('ood_ok'), '->', 'PASS' if (ood.get('ood_ok') and ood.get('n_distinct',0)>=18) else 'FAIL')
print('B3 floor:', 'PASS' if led and min(min(l) for l in led)>=0.049 else 'FAIL', '| clearings:', len(led))
print('B4 supply_resp:', r.get('supply_responsiveness'), '->', 'PASS' if (r.get('supply_responsiveness') or 0)>0.3 else 'FAIL')
print('B5 finite curve:', 'PASS' if all(c.get('val_ce') is not None and math.isfinite(c['val_ce']) for c in r.get('curve',[])) else 'FAIL')
print('alloc final:', r.get('mkt_final'))